# Train fastText 3-gram - Intent Classification

Huấn luyện fastText supervised cho 5 intent tiếng Việt không dấu. Test không được dùng trong train hoặc chọn hyperparameter.

In [1]:
# Nếu chưa có package: %pip install -q fasttext-wheel scikit-learn
from pathlib import Path
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

ROOT = Path.cwd()
DATA = ROOT / 'data'
if not DATA.exists():
    for parent in ROOT.parents:
        if (parent / 'data').exists():
            ROOT, DATA = parent, parent / 'data'
            break

df = pd.read_csv(DATA / 'intent_train.csv')
OUT = ROOT / 'fasttext_work'
OUT.mkdir(exist_ok=True)

def clean_text(text):
    # Giữ tiếng Việt không dấu, chuẩn hóa whitespace và ký tự đặc biệt.
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9_\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df['text_clean'] = df['question'].map(clean_text)
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['intent'], random_state=42)

def write_fasttext_file(frame, path):
    with open(path, 'w', encoding='utf-8') as f:
        for _, row in frame.iterrows():
            f.write(f"__label__{row.intent} {row.text_clean}\n")

train_path, val_path = OUT / 'train.txt', OUT / 'valid.txt'
write_fasttext_file(train_df, train_path)
write_fasttext_file(val_df, val_path)
print('train:', train_df.shape, 'valid:', val_df.shape)
print(df.intent.value_counts().sort_index())

train: (4800, 4) valid: (1200, 4)
intent
COUNT           1200
EXIST           1200
LIST            1200
LOCATION        1200
OUT_OF_SCOPE    1200
Name: count, dtype: int64


In [2]:
import fasttext

model = fasttext.train_supervised(
    input=str(train_path),
    lr=0.5,
    epoch=75,
    wordNgrams=3,
    dim=64,
    bucket=2000,
    minn=2,
    maxn=5,
    loss='softmax',
    thread=4,
)
vocab_size = len(model.get_words())
estimated_params = (vocab_size + 2000) * 64 + df['intent'].nunique() * 64
print(f'estimated parameters: {estimated_params:,}')
assert estimated_params <= 200_000, f'Parameter budget exceeded: {estimated_params:,}'
model_path = OUT / 'intent_fasttext_3gram_200k.bin'
model.save_model(str(model_path))
print('saved:', model_path)

estimated parameters: 145,088
saved: c:\Users\admin\OneDrive - The University of Technology\Desktop\edge_ai\fasttext_work\intent_fasttext_3gram_200k.bin


In [3]:
def predict_one(text):
    labels, scores = model.predict(clean_text(text), k=1)
    return labels[0].replace('__label__', ''), float(scores[0])

pred = [predict_one(x)[0] for x in val_df.question]
print('accuracy:', accuracy_score(val_df.intent, pred))
print('macro-F1:', f1_score(val_df.intent, pred, average='macro'))
print(classification_report(val_df.intent, pred, digits=4))

accuracy: 1.0
macro-F1: 1.0
              precision    recall  f1-score   support

       COUNT     1.0000    1.0000    1.0000       240
       EXIST     1.0000    1.0000    1.0000       240
        LIST     1.0000    1.0000    1.0000       240
    LOCATION     1.0000    1.0000    1.0000       240
OUT_OF_SCOPE     1.0000    1.0000    1.0000       240

    accuracy                         1.0000      1200
   macro avg     1.0000    1.0000    1.0000      1200
weighted avg     1.0000    1.0000    1.0000      1200

